# 05 — Reconnaître le RYTHME : Manjanin / Maraka / Woloso

On l'a vu en fondation : l'accompagnement jembé est un **squelette partagé**, mais le **dunun** a un motif propre à chaque pièce. On reconnaît donc le rythme à partir du **profil métrique du dunun** (probabilité de frappe sur les 12 subdivisions du cycle).

Tâche **supervisée**, évaluée en **leave-one-out** (chaque prise testée par un modèle entraîné sur les autres).

## 0. Imports & dataset (profil dunun par prise)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
import djembe_utils as dj

c = dj.list_corpus(); rows = []
for (piece, take), _ in c[c['code']=='D1'].groupby(['piece','take']):
    p = dj.subdivision_profile(piece, take, 'D1')
    if p is None:
        continue
    rows.append({'piece':piece, 'take':take, **{f'sd{i+1}':p.values[i] for i in range(12)}})
data = pd.DataFrame(rows)
SD = [f'sd{i+1}' for i in range(12)]
print(len(data), 'prises avec dunun'); display(data.groupby('piece').size().rename('n_prises'))

## 1. Les motifs de dunun, prise par prise
Chaque ligne = une prise. On voit les 3 pièces former des blocs distincts.

In [ ]:
d = data.sort_values('piece')
plt.figure(figsize=(9, 6))
plt.imshow(d[SD].values, aspect='auto', cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='proba de frappe du dunun')
plt.xticks(range(12), range(1,13)); plt.yticks(range(len(d)), d['piece'].values)
plt.xlabel('subdivision du cycle (1–12)'); plt.title('Motif de dunun par prise'); plt.show()

## 2. Classifieur (leave-one-out)

In [ ]:
pred = cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=0),
                         data[SD].values, data['piece'].values, cv=LeaveOneOut())
print('Accuracy leave-one-out :', round(accuracy_score(data['piece'], pred), 3))
display(pd.DataFrame(confusion_matrix(data['piece'], pred, labels=dj.PIECES),
                     index=['vrai '+p for p in dj.PIECES], columns=['préd '+p for p in dj.PIECES]))

## Bilan
- Le **motif du dunun** suffit à reconnaître le rythme avec une bonne précision (les confusions restantes viennent du petit nombre de prises).
- On utilise ici les onsets **annotés** ; pour passer à de l'audio brut, il faudrait d'abord détecter les onsets du dunun (cf. la séparation en archive).
- C'est la base de l'objectif final : reconnaître un rythme → puis le **suivre et le générer** (étape ∞).